# 🇧🇩 Fine-Tuning Bangla TrOCR on Kaggle GPU (T4 / P100)

This notebook fine-tunes a **Cross-Lingual Vision-Encoder-Decoder model** for handwritten Bangla document recognition:
- **Visual Encoder**: `google/vit-base-patch16-224-in21k`
- **Language Decoder**: `sagorsarker/bangla-bert-base`
- **Target**: Recognizes complex cursive strokes and **Juktakkhor (যুক্তাক্ষর)**.

---
### ⚙️ Kaggle Settings Before Running:
1. **Accelerator**: Select **GPU T4 x2** or **GPU P100** (Right sidebar -> Session options -> Accelerator).
2. **Internet**: Toggle **Internet ON** (Right sidebar -> Session options -> Internet).

In [ ]:
# 1. Environment & Dependencies
!nvidia-smi
!pip install -q transformers datasets jiwer accelerate torchvision openpyxl

In [ ]:
# 2. Locate BN-HTRd Dataset in /kaggle/input
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

input_root = Path("/kaggle/input")
print("Scanning /kaggle/input for BN-HTRd datasets...")
for p in input_root.iterdir():
    print(f"  Found dataset: {p.name}")

# Check for Excel files
xlsx_files = list(input_root.rglob("*.xlsx"))
print(f"Total Excel annotation files found: {len(xlsx_files)}")
if not xlsx_files:
    raise FileNotFoundError("No .xlsx files found in /kaggle/input. Please attach the BN-HTRd dataset.")

In [ ]:
# 3. Build Word-Image Mapping Manifest
records = []
missing = 0

kaggle_gt_candidates = list(input_root.glob("**/Recognition_Ground_Truth_Texts"))

if kaggle_gt_candidates:
    gt_dir = kaggle_gt_candidates[0]
    words_dir = gt_dir.parent / "Segmentation_Images" / "Words"
    all_xlsx = list(gt_dir.rglob("*.xlsx"))
    print(f"Detected standard Kaggle BN-HTRd structure with {len(all_xlsx)} Excel files.")
    for xlsx_path in tqdm(all_xlsx, desc="Parsing Excel files"):
        try:
            df = pd.read_excel(xlsx_path)
            if 'Id' not in df.columns or 'Word' not in df.columns:
                continue
            for _, row in df.iterrows():
                crop_id = str(row['Id']).strip()
                word_text = str(row['Word']).strip()
                if not crop_id or not word_text or word_text == 'nan':
                    continue
                parts = crop_id.split('_')
                if len(parts) != 4:
                    continue
                doc_id = parts[0]
                page_dir = f"{parts[0]}_{parts[1]}"
                line_dir = f"{parts[0]}_{parts[1]}_{parts[2]}"
                img_p = words_dir / doc_id / page_dir / line_dir / f"{crop_id}.jpg"
                if not img_p.exists():
                    img_p = words_dir / doc_id / page_dir / line_dir / f"{crop_id}.JPG"
                if img_p.exists():
                    records.append({"image_path": str(img_p), "text": word_text})
                else:
                    missing += 1
        except Exception as e:
            pass
else:
    print(f"Parsing direct document folder structure across {len(xlsx_files)} Excel files...")
    for xlsx_path in tqdm(xlsx_files, desc="Parsing documents"):
        try:
            df = pd.read_excel(xlsx_path)
            if 'Id' not in df.columns or 'Word' not in df.columns:
                continue
            doc_dir = xlsx_path.parent
            words_dir = doc_dir / "Words"
            for _, row in df.iterrows():
                crop_id = str(row['Id']).strip()
                word_text = str(row['Word']).strip()
                if not crop_id or not word_text or word_text == 'nan':
                    continue
                parts = crop_id.split('_')
                if len(parts) != 4:
                    continue
                page_dir = f"{parts[0]}_{parts[1]}"
                img_p = words_dir / page_dir / f"{crop_id}.JPG"
                if not img_p.exists():
                    img_p = words_dir / page_dir / f"{crop_id}.jpg"
                if not img_p.exists():
                    line_dir = f"{parts[0]}_{parts[1]}_{parts[2]}"
                    img_p = words_dir / page_dir / line_dir / f"{crop_id}.JPG"
                    if not img_p.exists():
                        img_p = words_dir / page_dir / line_dir / f"{crop_id}.jpg"
                if img_p.exists():
                    records.append({"image_path": str(img_p), "text": word_text})
                else:
                    missing += 1
        except Exception as e:
            pass

print(f"\nSuccessfully mapped {len(records)} word-image pairs! (Missing: {missing})")
manifest_csv = "/kaggle/working/train_manifest.csv"
pd.DataFrame(records).to_csv(manifest_csv, index=False, encoding="utf-8")
print(f"Manifest saved to: {manifest_csv}")

In [ ]:
# 4. Initialize Vision-Encoder-Decoder Model (ViT + Bangla BERT)
import torch
from transformers import (
    ViTImageProcessor,
    AutoTokenizer,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator
)
from datasets import Dataset
from PIL import Image
import numpy as np
import jiwer

encoder_id = "google/vit-base-patch16-224-in21k"
decoder_id = "sagorsarker/bangla-bert-base"

print("Loading ViT Image Processor and Bangla BERT Tokenizer...")
processor = ViTImageProcessor.from_pretrained(encoder_id)
tokenizer = AutoTokenizer.from_pretrained(decoder_id)

print("Initializing VisionEncoderDecoderModel...")
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(encoder_id, decoder_id)

# TrOCR token configuration
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Generation config
model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.sep_token_id
model.generation_config.max_length = 32
model.generation_config.num_beams = 1
model.generation_config.early_stopping = False
model.generation_config.no_repeat_ngram_size = 3
print("Model architecture configured successfully!")

In [ ]:
# 5. Dataset Transforms and Evaluation Metric
df = pd.read_csv(manifest_csv).dropna()
print(f"Total available samples in dataset: {len(df)}")

# (Optional) To train on a subset for quick testing, uncomment the line below:
# df = df.sample(n=25000, random_state=42).reset_index(drop=True)

hf_dataset = Dataset.from_pandas(df)
hf_dataset = hf_dataset.train_test_split(test_size=0.05, seed=42)
train_ds = hf_dataset['train']
eval_ds = hf_dataset['test']
print(f"Training samples: {len(train_ds)}, Validation samples: {len(eval_ds)}")

def preprocess_batch(batch):
    images = [Image.open(p).convert("RGB") for p in batch["image_path"]]
    pixel_values = processor(images, return_tensors="pt").pixel_values
    
    labels = tokenizer(
        batch["text"], 
        padding="max_length", 
        max_length=64, 
        truncation=True
    ).input_ids
    
    # Mask padding tokens with -100 so they are ignored in cross-entropy loss
    labels = [
        [token if token != tokenizer.pad_token_id else -100 for token in label]
        for label in labels
    ]
    
    batch["pixel_values"] = pixel_values
    batch["labels"] = labels
    return batch

train_ds.set_transform(preprocess_batch)
eval_ds.set_transform(preprocess_batch)

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]
    labels_ids = np.where(labels_ids != -100, labels_ids, tokenizer.pad_token_id)
    pred_ids = np.where(pred_ids != -100, pred_ids, tokenizer.pad_token_id)
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    labels_str = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)
    cer = jiwer.cer(reference=labels_str, hypothesis=pred_str)
    return {"cer": cer}

In [ ]:
# 6. Start Training with FP16 on Kaggle GPU
output_dir = "/kaggle/working/trocr_bangla"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    predict_with_generate=True,
    eval_strategy="epoch",
    remove_unused_columns=False,
    per_device_train_batch_size=16,   # T4 GPU (16GB VRAM) easily accommodates batch size 16
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    fp16=True,                        # Mixed-precision FP16 enabled for 2x faster training
    dataloader_num_workers=2,
    logging_steps=50,
    save_strategy="epoch",
    num_train_epochs=3,               # 3 to 5 epochs
    save_total_limit=2,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=default_data_collator,
)

print("Starting Seq2Seq training on Kaggle GPU...")
trainer.train()

print(f"Saving final model and processors to {output_dir}...")
trainer.save_model(output_dir)
processor.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Training complete and model saved!")

In [ ]:
# 7. Compress Trained Model for Easy Download
import shutil

zip_path = "/kaggle/working/trocr_bangla_trained"
print(f"Compressing {output_dir} into {zip_path}.zip...")
shutil.make_archive(zip_path, 'zip', output_dir)
print("Zip archive created successfully!")
print("\nDownload 'trocr_bangla_trained.zip' from the Output section in the Kaggle right-hand panel.")